<a href="https://colab.research.google.com/github/ChandrikaImmadi/GenAI-Tasks/blob/main/Week2b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu chromadb tiktoken

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import chromadb
import tiktoken

In [ ]:
dataset = """
The Eiffel Tower is a wrought-iron lattice tower in Paris, France. It was constructed from 1887 to 1889 as the entrance to the 1889 World's Fair.
Python is a high-level programming language known for its readability and versatility. It is widely used in web development, data science, and AI.
FAISS is a library developed by Facebook AI Research for efficient similarity search and clustering of dense vectors.
Chroma is an open-source embedding database designed for AI applications, enabling developers to store and query embeddings easily.
Sentence Transformers allow easy computation of dense vector representations for text, enabling semantic search and clustering.
The Great Wall of China is a historic fortification built to protect Chinese states against invasions.
Machine learning enables systems to learn patterns from data without explicit programming, powering modern AI applications.
Mount Everest is the highest mountain above sea level, located in the Himalayas, attracting climbers from around the world.
"""

In [ ]:
def chunk_text(text, min_tokens=200, max_tokens=500, model_name="gpt2"):
    enc = tiktoken.get_encoding(model_name)
    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk = enc.decode(tokens[start:end])
        if len(enc.encode(chunk)) >= min_tokens:
            chunks.append(chunk)
        start = end
    return chunks

chunks = chunk_text(dataset)
print(f"Created {len(chunks)} chunks")

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks)

In [ ]:
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(np.array(embeddings))

In [ ]:
client = chromadb.Client()
collection = client.create_collection(name="wiki_chunks")
collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=[str(i) for i in range(len(chunks))]
)

In [ ]:
def retrieve(query, top_k=3):
    query_emb = model.encode([query])

    # FAISS search
    D, I = faiss_index.search(query_emb, k=top_k)
    faiss_results = [chunks[idx] for idx in I[0]]

    # Chroma search
    chroma_results = collection.query(
        query_texts=[query],
        n_results=top_k
    )["documents"][0]

    return {
        "FAISS": faiss_results,
        "Chroma": chroma_results
    }

In [ ]:
results = retrieve("What is the tallest mountain?", top_k=3)
print("FAISS Results:\n", results["FAISS"])
print("\nChroma Results:\n", results["Chroma"])